# Instructions For Adding A Tool

This Notebook will take you through the standard process of adding a tool to the PyCVA package. The following instructions are relevant to development on Windows and have not yet been tested for Linux nor Mac.

## 1. Conda Environment
The first step will be to make your medical tool work within a conda environment (This is done so that the Docker image created later can be made with a standard and easy to implement procedure not unique to any tool). If you don't already have Conda go here https://docs.conda.io/en/latest/ and follow the instructions for it's installation.

Once you have setup up Conda, make sure to install the medical tool's source code locally. This is usually done by accessing the tools GitHub page and downloading/cloning from there onto your computer.

Once these two requirements are finished, create a conda envirnoment and make sure the tool works as intended within a Conda environment. To create a conda environment please use the command below in your terminal and name the environment something relevant to your tool, like its name for instance.





In [ ]:
# This is not python executable and should be copied into your terminal
conda create -n your_env_name

# This usually involves pulling the source code and installing the dependencies w/in the Conda environment.
pip install -r requirements.txt # used to install pip requirements from a .txt file. 

### 1.1 Zenodo

If there are any model weights, we dont want them to be added to the Docker image created for the tool as they will bloat it unnecessarrily. Any weights present within the project must be uploaded to Zenodo and pulled at runtime. This technique is referred to as 'lazy loading' and helps to ensure that later the end user only downloads what they need when they need it.

First upload the model weights to Zenodo by following this procedure https://help.zenodo.org/docs/deposit/create-new-upload/. Then copy the download all link as save it for later.

<img src="Adding_A_Tool_Artefacts/Zenodo_Page.png" alt="Alt text" width="300"/>

To install the weights at run times it is advised to copy the method below and paste it within your tools main executable file. Eg. for the tool CAVE this would be the 'predict.py' file. Once you have this method within the appropriate Python file, it must be called with the Zenodo link to the download weights (mentioned previously), and the folder location where you wish for the weights to be installed during runtime for your specific tool.

This must be done before any of the machine learning functionality that requires the weights is used with the Python file.

 




In [ ]:
# This code is python code that can be copied and pasted into the main method of the medical tool. 
from urllib.parse import urlparse, unquote
import os
import urllib

def download_files(url_list, folder):
    if not os.path.exists(folder):
        os.makedirs(folder)
    
    if not os.listdir(folder):
        for url in url_list:
            parsed_url = urlparse(url)
            filename = os.path.basename(parsed_url.path)
            filename = unquote(filename)

            local_path = os.path.join(folder,filename)
            urllib.request.urlretrieve(url, local_path)
    else:
        print(f"Model folder already poulated at {folder}")


Below is an example of how to use the aforementioned method for copying the urls into the MODELS_DIR path. 

 <img src="Adding_A_Tool_Artefacts/Example_of_Download.png" alt="Alt text" width="780"/>


## 2. Export the Conda Environment

Once the medical tool is working as intended within the conda environment and the model weights are lazy loaded using zenodo and the "download_files" method. The environment can be exported. This makes a very detailed and precise YAMAL file that can later be used in conjunction with Docker to recreate the exact conditions under which the tool can thrive but with the added bonus of full encapsulation. To do this activate the Conda environment in which the tool works as intended. If you have forgotten the name of your conda environment run the following command to get a list: 

In [ ]:
conda info --envs # Gets you a list of all the environments on the current syste

conda activate env_name # Activates the conda environment

Once inside the right Conda environment, run the following command:

In [ ]:
# "environment.yml" is the name of the file that will be created that will contain the canda environment

conda env export > environment.yml

This will give you a file that represents all the correct dependencies for your tool and will be used in the next step to create the Docker file

# 3. Docker Image

Thanks to our newly created YAMAL file we can easily create a Dockerfile around the Conda environment. We need to create a file in the source code (Tool_Folder/Dockerfile) of the medical tool called Dockerfile. Within the Dockerfile, paste the code below and fill in the environment name you want to create,

In [ ]:
FROM continuumio/miniconda3


ENV ENV_NAME {{_env_name}}

RUN apt-get update && apt-get install -y \
    libpng-dev \
    libtiff-dev \
    libjpeg-dev \
    libgl1-mesa-glx \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

COPY {{YAMAL_file_name}}.yml .

RUN conda env create -n $ENV_NAME -f {{_YAMAL_file_name}}.yml && conda clean -afy

ENV PATH /opt/conda/envs/$ENV_NAME/bin:$PATH

WORKDIR /app

COPY . .

In order to test the Dockerfile and all the functionality inside you can use Docker Compose. This makes launching the serivce much easier. A typical Compose file looks something like the code displayed below. It essentially takes an image, the volumes you want to mount and the cmd line to execute. The entry point is the python executable file for the functionality you are trying to access.

In [ ]:

services:
  autotici:
    image: [your_image_name_here]
    build:
      context: .
      dockerfile: Dockerfile
    entrypoint: ["conda", "run", "--no-capture-output", "-n", "name_env", "python", "EntryScript.py"]
    volumes:
      - any_volumes_you_wish_to_mount
    working_dir: /app
    command:
      - "The command you wish to run inside the container"

#Example with Phase_Predict
  phase_predict:
    image: autotici_docker
    build:
      context: .
      dockerfile: Dockerfile
    entrypoint: ["conda", "run", "--no-capture-output", "-n", "autotici_env", "python", "phase_classification/phase_predict.py"]
    volumes:
      - ./input_images:/app/input_images
      - ./output:/app/output
      - ./models:/app/models
    command: ["input_images/SN3_Vap_SOP1.2.826.0.1.3680043.9.6827.2401407280984092171337075126937802959.dcm"]


The above builds the Docker image an runs the functionality specified in the command section. This will be an image unique to your architecture. If you want to create an image for each type of architecture automatically, buildx can be used. The insturctions for which can be found here: https://docs.docker.com/reference/cli/docker/buildx/build/

# 4. Upload

Once the docker image is properly created, you need to push it to the Dockerhub account associated with this package. This can be easily done using Docker commands shown below. It is recommended that you tag the image to a certain version rather than 'latest' as it then properly freezes your functionality rather than pulling potential updated images that with ':latest' that may have different functionaity inside and break your package.

In [ ]:
#login to the account associated with the package
docker login

#replace with the relevant local name, name and tag
docker tag local_image_name your_dockerhub_username/image_name:tag

#finally push the image 
docker push your_dockerhub_username/repo_name:tag

Remember the name of the directory, image name and tag for later when it is referenced in order to pull the image from Dockerhub within the wrapper class. This will be explained later though. 


For our purposes of exemplifying the process we will use the 
"docker push your-dockerhub-username/repo-name:tag" later on in the code for consistency

# 4.2 Setting up the tools folder

Before we start with the wrapper class step. You have to setup the tool within the context of the package. To do this create a folder within the PyCVA/src/pycva directory and call it the name of your tool. Within this folder you will create two files. 

1. tool_name.py
2. tool_name_wrapper.py

This will set you up nicely for the steps to come:

 <img src="Adding_A_Tool_Artefacts/Directory_Structure.png" alt="Alt text" width="200"/>


# 5. Wrapper class

This is the class that will be translating Python into commands for the Docker image to execute and is called a wrapper class. In order to create a consistent implementation process for these wrapper classes they all extend an abstract super class which requires them to implement two methods. One method for executing docker commands and one method for executing singularity commands.

To get started, the first step is understanding the input and outputs of your containerised tool. Once you have chosen those you need to decide whether you will mount weights or not. This is heavily recommended for any inputs, outputs and models as not mounting these weights will cause issues like image bloat or the image not picking up the input and output directories.

Below is a template off which you can fill in with the information relevant to your containerised tool. You will need your docker repository link for the image (from docker hub). You will also come up with a name for the potential sif (singularity file) that is created. Finally you must fill in the paramters with those relevant to your tool and construct the docker command. 



In [ ]:
import os
import subprocess
from pycva.common.containerised_tool_base import ContainerisedToolBase
import sys

class your_tool_name_wrapper(ContainerisedToolBase):


    def __init__(self):
        
        # Initializes the Your_Tool_Name_Wrapper with Docker and Singularity configurations.
        docker_image = "your-dockerhub-username/repo-name:tag" # When you have finalised your image, remeber that latest is a bad tag to have. Go with something solid
        sif_name = "Your_Tool_Name.sif"
        super().__init__(docker_image, sif_name)

    def _run_docker(self, pre_image, output_dir,options='A'):
       
        # Runs Your_Tool_Name via Docker.

        # Args:
        #     pre_image (str): Path to the PreEVT DICOM file.
        #     option (str): Type of execution (A, B, C)
        #     output_dir (str): Path to the output directory.

        # Raises:
        #     subprocess.CalledProcessError: If Docker command execution fails.
        
        os.makedirs(output_dir, exist_ok=True)
        docker_cmd = [
            "docker", "run", "--rm",
            "-v", f"{os.path.abspath(os.path.dirname(pre_image))}:/app/pre", # mounting the folder contianing the file (For inputs we want the specfic file)
            "-v", f"{os.path.abspath(output_dir)}:/app/output" # mounting the whole output dir (For outputs we just need the output dir)
        ]
        docker_cmd += [
            self.docker_image,
            "python", "Your_Entry_Point_.py",
            f"/app/pre/{os.path.basename(pre_image)}",
            options
        ]
        # Any True/False values can be added like so:
        # if motion_correction: 
        #     docker_cmd.append("-m")
        # if preregistration:
        #     docker_cmd.append("-l")
        # if view:
        #     docker_cmd += ["-v", view]
        docker_cmd += ["-o", "/app/output"]

        print(f"[DOCKER] Running: {' '.join(docker_cmd)}")
        subprocess.run(docker_cmd, check=True)

    def _run_singularity(self, pre_image, output_dir, options='A'):
        
        os.makedirs(output_dir, exist_ok=True)
        binds = [
            f"{os.path.abspath(os.path.dirname(pre_image))}:/app/pre",
            f"{os.path.abspath(output_dir)}:/app/output"
        ]
        for b in binds:
            singularity_cmd += ["--bind", b]

        singularity_cmd += [
            self.sif_path,
            "python", "Your_Entry_Point_.py",
            f"/app/pre/{os.path.basename(pre_image)}",
            options
        ]
        # This is the same as with a docker cmd
        # if motion_correction:
        #     singularity_cmd.append("-m")
        # if preregistration:
        #     singularity_cmd.append("-l")
        # if view:
        #     singularity_cmd += ["-v", view]
        singularity_cmd += ["-o", "/app/output"]

        print(f"[SINGULARITY] Running: {' '.join(singularity_cmd)}")
        subprocess.run(singularity_cmd, check=True)

    def run(self, *args, **kwargs):
        # Runs AutoTICI using either Docker or Singularity, depending on availability.

        # This method delegates to `_run_docker` or `_run_singularity` based on the
        # runtime environment.

        # Args:
        #     *args: Positional arguments passed to the container execution method.
        #     **kwargs: Keyword arguments passed to the container execution method.

        # Raises:
        #     RuntimeError: If neither Docker nor Singularity is available.
        
        return super().run(*args, **kwargs)


This process can be followed for any functionality within the container. Lets say for example you have two pieces of functionality you want to access with two different methods. In this case you can make two wrapper classes that call the same image. This can be seen with phase predict and autoTICI. 

When doing this however, remember to adjust your functionality to facilitate the outputs. For instance with phase predict, traditionally the outputs where sent to the terminal. To make it an effective piece of containerised functionality we generally want to export the output to a file to facilitate automation and reuse of said output later on. 

# 6. Integrated Package (Optional)

Within this package we want to be able to use two types of functionalities. 

1. Containerised 
2. Pure Python Util Methods

This is because, hidden within most of these tools are some very useful methods that when paired with other tools, can act as very useful preprocessing functionalities. 

This step is however completely optional and does add a fair amount of complexity. SKIP TO STEP 7 if you just need the containerised functionality.

Another benefit of offering this split is because through "lazy loading" (a technique wherein I only install what is needed at the moment), the user never has to install an image or weights and can instead just use the light weight python methods from the source code. 

To get started, locate the folder for your tool. It will be located within the PyCVA/src/pycva directory of the package. Once there take the python classes containing the functionality you want to integrate and copy them into your tools folder. 

Once in there take a look at the imports (located at the top of the file you just copied in) 

<img src="Adding_A_Tool_Artefacts/Imports_Example.png" alt="Alt text" width="600"/> 


Once you know the imports, go to the requirements.txt or environment.yml file located within your tools source code (the code you pulled from GitHub originally) and match the imports to the requirements to accertain the exact requirements needed for your new functionality to work within the package. Then create a test_requirements.txt file with both the new tools dependencies and the packages dependencies inside. A big part of this process hinges on the functionality you are importing working with the dependencies already present within the package and so testing before integration is an important step. This will test whether your tools works with python 3.11 and whether the dependencies will clash.


 


In [ ]:

# Make sure the test_requirements.txt and any test file you create are within your source code so it can properly reach the methods its trying to test.

# First make sure you have python 3.11
python --version

# Make sure you are in a directory where you are happy for the .venv folder to be placed
cd /path/to/your/project

# Create the venv
python -m venv venv

# Execute the following in command line to activate the venv
venv\Scripts\activate.bat

# Once inside the venv update your pip
pip install --upgrade pip

# Finally you can install the package and new tool deps into your empty venv
pip install -r test_requirements.txt




Hopefully this works. If it installs everything first time this is a good sign otherwise you need to try and make your dependencies work with the current packages'. You can also run a python executable with all the imports needed for the new tools functionality to see if you get an error.

Within your project's source code you can also create a test.py file that does smoke tests on all the new methods you have added. 

A smoke test just runs the method and asserts that the result is not null. An example can be found below. 

In [ ]:
import pydicom
import sys, os

project_root = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
sys.path.append(project_root)

from utils.utils import * # The class being referenced and smoke tested

ds = pydicom.dcmread("R0160/SN4_Vap_SOP1.3.6.1.4.1.40744.9.96512919107112378459801894939320275800.dcm")
img = ds.pixel_array

def test_normalize():
    result = normalize(img)
    assert result is not None

def test_normalize_0_1():
    result = normalize_0_1(img)
    assert result is not None

def test_remove_text_and_border():
    result = remove_text_and_border(img)
    assert result is not None

def test_minip():
    seq = np.stack([img]*5, axis=0)
    result = minip(seq, axis=0)
    assert result.shape == img.shape

def test_read_sequence():
    seq, spacing = read_sequence(ds.filename)
    assert seq is not None
    assert spacing is not None

def test_get_pixel_spacing_from_header():
    spacing = get_pixel_spacing_from_header(ds)
    assert spacing is not None

Once you know for sure that the functionality integrates smoothly with the package you can go to the pyproject.toml file and add your new test_requirements.txt dependencies to it. This will update all the projects dependencies so be very sure that you are happy with how everything is functioning before doing this. 

In [ ]:
# while in the venv you created, freeze the requirements

pip freeze # this will just print the current dependencies in the terminal where you can copy and paste them elsewhere.

# or

pip freeze > new_requirements.txt # If you want the current requirements to export to a new requirements.txt


# This will reveal your new package requirements which you must copy into the pyproject.toml file dependencies section with the format shown below.

<img src="Adding_A_Tool_Artefacts/pyproject_toml.png" alt="Alt text" width="650"/>

# 7. Tool Wrapper

Now that you have your translater class and your pure python functionality you need to "unify the interface" for the user into one object. This essentially means abstracting all functionality into a single class/object in which the user can call to access all the functionality from one place. This allows for the user to use both pure python and contianerised functionality without changing how they interact with the tool.


In [ ]:

from . import autotici_wrapper # import any container wrappers you want to call with this object

from .utils import utils # import any pure python classes you want to call with this object
import sys
import os

class autotici():

    def __init__(self):
        
        # Initializes the AutoTICI as none to allow lazy loading later on.
        
        self._autotici_wrapper = None  # This makes sure when you start the object it doesn't imediately download the image and weights

    @property
    def autotici_wrapper(self):
        
        # This method ONLY runs when someone accesses self.autotici_wrapper
        
        if self._autotici_wrapper is None:
            self._autotici_wrapper = autotici_wrapper.autotici_wrapper()
        return self._autotici_wrapper


# Containerised functionality
    def run_autoTICI(self, *args, **kwargs):

        return self.autotici_wrapper.run(*args, **kwargs) # At this point it will start the container download



# Utils.py
    def normalize(self, img):
         
         return utils.normalize(img) # Pure python methods that can be accessed without fussing about the containers


    def normalize_0_1(self, img):
         
         return utils.normalize_0_1(img)


   

As seen above making the tool_wrapper is just importing all the functionality you want this tool to represent and the make methods that return the functionality made elsewhere. 

The most complicated part is the properties but you can just copy what is in the above snippet and replace with the name of the containerised tool you are implementin.

With all the implementation now inplace, you can finally load the new package with the commands below.

In [ ]:
cd PyCVA # makes sure you are 1 folder up from src

pip install e . # this will rebuild your package with the current setup.py and dependencies locally where you can do further tests.

In order to upload to PyPi you will need to follow the instructions laid out in the docs: https://packaging.python.org/en/latest/tutorials/packaging-projects/